# 06 — Full Evaluation

Loads best fusion model checkpoint, runs full benchmark evaluation:
- Optimal threshold selection (maximize sensitivity-weighted F1)
- All classification + clinical metrics with bootstrap 95% CI
- Modality ablation study
- Baseline comparison (LR, RF, GBM, XGBoost on HPO features)
- Generates `reports/benchmark_report.md`

**Run order: 01 → 02 → 03 → 04 → 05 → 06**

In [ ]:
# CELL 2: Mount Google Drive (checkpoints will be saved here after training)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/EarlyMind/checkpoints', exist_ok=True)
print('Drive mounted. Fresh .pt files will be saved to MyDrive/EarlyMind/checkpoints/')

In [ ]:
# CELL 1: Clone / pull repo
import os
if not os.path.exists('/content/earlyMind'):
    !git clone https://github.com/Rickykapoor/earlyMind.git /content/earlyMind
%cd /content/earlyMind
!git pull origin main

In [ ]:
# CELL 2: Install dependencies
!pip install -q mne nibabel nilearn openneuro-py torch torchvision \
  streamlit plotly scikit-learn pytorch-tabnet \
  xgboost catboost tqdm pyyaml scipy

In [ ]:
# CELL 3: Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 4: Download fusion model checkpoint from GitHub Releases
# This is produced by notebook 05. Download from Releases if not present.
import os

RELEASE = 'https://github.com/Rickykapoor/earlyMind/releases/download/v1.0.0-data'

os.makedirs('checkpoints', exist_ok=True)

for ckpt in [
    'fusion_model.pt',
    'eeg_encoder_pretrained.pt',
    'mri_encoder_pretrained.pt',
    'hpo_encoder_pretrained.pt',
]:
    path = f'checkpoints/{ckpt}'
    if not os.path.exists(path):
        print(f'Downloading {ckpt} ...')
        !wget -qO {path} {RELEASE}/{ckpt}
    else:
        print(f'{ckpt} already present. Skipping.')

# Download processed datasets if missing
for name, tar, dest in [
    ('eeg', 'eeg_raw.tar.gz',    'datasets/eeg'),
    ('mri', 'mri_raw.tar.gz',    'datasets/mri'),
    ('hpo', 'facial_raw.tar.gz', 'datasets/facial'),
]:
    if not os.path.exists(f'datasets/processed/{name}') and not os.path.exists(dest):
        !wget -qO {tar} {RELEASE}/{tar}
        !tar -xzf {tar} -C {dest}

print('All assets ready.')

In [ ]:
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 5: Setup
import sys
sys.path.insert(0, '/content/earlyMind')

import torch
import numpy as np
from src.config import cfg

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg.paths.makedirs()

ckpt_path = cfg.paths.checkpoints / 'fusion_model.pt'
if not ckpt_path.exists():
    raise FileNotFoundError(f'Checkpoint missing: {ckpt_path}. Run notebook 05 first or re-run Cell 4.')
print(f'Checkpoint found: {ckpt_path}')

In [ ]:
# Restore checkpoint from Drive if local copy is missing/corrupt
import shutil, os
_drive_ckpt = '/content/drive/MyDrive/EarlyMind/checkpoints/fusion_model.pt'
_local_ckpt = str(ckpt_path)
if os.path.getsize(_local_ckpt) < 1_000_000 and os.path.exists(_drive_ckpt):
    print('Local checkpoint corrupt — restoring from Drive...')
    shutil.copy2(_drive_ckpt, _local_ckpt)
    print(f'Restored: {os.path.getsize(_local_ckpt)/1e6:.0f} MB')
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 6: Load model
from src.models.fusion_model import build_fusion_model

n_hpo = cfg.model.hpo_n_features
model = build_fusion_model(n_hpo=n_hpo)
model.load_state_dict(torch.load(str(ckpt_path), map_location='cpu', weights_only=False))
model = model.to(device)
model.eval()
print('Model loaded.')

In [ ]:
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 7: Build DataLoaders
from src.data.fusion_dataset import build_dataloaders

_, val_ldr, test_ldr = build_dataloaders(
    eeg_processed_dir  = cfg.paths.eeg_processed,
    mri_processed_dir  = cfg.paths.mri_processed,
    hpo_processed_dir  = cfg.paths.hpo_processed,
    eeg_raw_dir        = cfg.paths.eeg_raw,
    batch_size         = 32,
    modality_dropout_p = 0.0,    # no dropout at eval
)
print(f'Val batches: {len(val_ldr)}, Test batches: {len(test_ldr)}')

In [ ]:
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 8: Run inference and compute metrics
from src.training.evaluate import run_inference, find_optimal_threshold, compute_classification_metrics, bootstrap_auc_ci

val_res  = run_inference(model, val_ldr,  device)
test_res = run_inference(model, test_ldr, device)

thr = find_optimal_threshold(val_res['probs'], val_res['labels'])
print(f'Optimal threshold: {thr:.3f}')

metrics = compute_classification_metrics(
    test_res['probs'], test_res['labels'],
    pred_dq=test_res['pred_dq'], true_dq=test_res['true_dq'],
    threshold=thr
)
auc_lo, auc_hi = bootstrap_auc_ci(test_res['probs'], test_res['labels'])

print('\n=== Test Set Metrics ===')
for k, v in metrics.items():
    print(f'  {k:15s}: {v}')
print(f'  AUC 95% CI: [{auc_lo:.3f}, {auc_hi:.3f}]')

In [ ]:
# CELL 9: ROC curve
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as auc_fn

fpr, tpr, _ = roc_curve(test_res['labels'], test_res['probs'])
roc_auc = auc_fn(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'EarlyMind AUC = {roc_auc:.3f}')
plt.plot([0,1],[0,1],'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — EarlyMind')
plt.legend()
plt.tight_layout()
plt.savefig(str(cfg.paths.reports) + '/roc_curve.png', dpi=100)
plt.show()

In [ ]:
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 10: Ablation study
from src.training.evaluate import ablation_study

ablation_df = ablation_study(model, test_ldr, device)
print('\n=== Ablation Study ===')
display(ablation_df)

In [ ]:
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 11: Baseline comparison on HPO features
from src.training.evaluate import run_baselines

X = np.load(str(cfg.paths.hpo_processed / 'hpo_matrix.npy'))
y = np.load(str(cfg.paths.hpo_processed / 'hpo_labels.npy'))

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

baseline_df = run_baselines(X_tr, y_tr, X_te, y_te)
print('\n=== Baselines ===')
display(baseline_df)

In [ ]:
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 12: Generate full benchmark report
from src.training.evaluate import generate_benchmark_report

report_path = generate_benchmark_report(
    model=model,
    test_loader=test_ldr,
    val_loader=val_ldr,
    device=device,
    X_train_hpo=X_tr, y_train_hpo=y_tr,
    X_test_hpo=X_te,  y_test_hpo=y_te,
    report_dir=str(cfg.paths.reports),
)

with open(report_path) as f:
    print(f.read()[:2000])  # Preview first 2000 chars

In [ ]:
# LAST CELL: Save fresh checkpoints to Google Drive
# These .pt files are too large for GitHub — save to Drive, then download
# to your Mac and upload to GitHub Releases (v1.0.0-data).
import shutil, os
drive_dir = '/content/drive/MyDrive/EarlyMind/checkpoints'
os.makedirs(drive_dir, exist_ok=True)

saved = []
for ckpt in ['fusion_model.pt']:
    src = f'checkpoints/{ckpt}'
    dst = f'{drive_dir}/{ckpt}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  Saved {ckpt} ({size_mb:.0f} MB) → Google Drive')
        saved.append(ckpt)
    else:
        print(f'  MISSING: {ckpt}')

print(f'\n{len(saved)} checkpoint(s) saved to Google Drive.')
print('Evaluation complete. All checkpoints in Drive. Upload fusion_model.pt to GitHub Releases.')
print('Next step: Download from Drive → Mac → Upload to GitHub Releases v1.0.0-data')
